In [1]:
import sys

from langchain_ollama import ChatOllama

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

from langchain_core.messages import HumanMessage, AIMessage

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)

# Change your imports to this:
from langchain_classic.chains import (
    create_history_aware_retriever,
    create_retrieval_chain,
)

from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)

# ==========================================
# 1. LLM Setup
# ==========================================

print("🔄 Initializing Ollama (Llama 3.1)...")

llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.3,
    timeout=60,
)

# ==========================================
# 2. Embedding Model
# ==========================================

print("🔄 Loading Embedding Model...")

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# ==========================================
# 3. Load ChromaDB
# ==========================================

print("🔄 Connecting ChromaDB...")

vector_db = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
)

retriever = vector_db.as_retriever(
    search_kwargs={"k": 3}
)

# ==========================================
# 4. Chat History
# ==========================================

chat_history = []

# ==========================================
# 5. Question Rewriter Prompt
# ==========================================

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Given a chat history and the latest user question,
which may reference previous conversation,
rewrite the question into a standalone question.

Do NOT answer the question.
Only rewrite it if needed.
            """,
        ),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# ==========================================
# 6. History Aware Retriever
# ==========================================

history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_q_prompt,
)

# create_history_aware_retriever() answer তৈরি করে না। db thake vector fatch kore 
# Question
#    ↓
# Better Question
#    ↓
# Documents
# create_history_aware_retriever() -> make better quenstion if needed -> bring information from db

# ==========================================
# 7. QA Prompt
# ==========================================

qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a helpful assistant.

Answer ONLY from the provided context.

If the answer is not available in the context,
reply with:

"I don't know based on the provided context."

Context:
{context}
            """,
        ),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# ==========================================
# 8. Document Chain
# ==========================================

question_answer_chain = create_stuff_documents_chain(
    llm,
    qa_prompt,
)
#create_stuff_documents_chain() নিজে create_history_aware_retriever থেকে কিছু retrieve করে না। এটা শুধু ধরে নেয় যে {context} already কেউ দিয়ে দিয়েছে।



# ==========================================
# 9. Final RAG Chain
# ==========================================

rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain,
)
#এখানে create_retrieval_chain() মাঝখানে glue হিসেবে কাজ করে।

# create_retrieval_chain automatically create_history_aware_retriever
# থেকে পাওয়া documents কে context হিসেবে
# create_stuff_documents_chain এ pass করে।

# ==========================================
# 10. Ask Function
# ==========================================

def run_rag_pipeline(user_question):

    print(f"\n📝 User: {user_question}")

    response = rag_chain.invoke(
        {
            "input": user_question,
            "chat_history": chat_history,
        }
    )

    answer = response["answer"]

    # Save history
    chat_history.append(
        HumanMessage(content=user_question)
    )

    chat_history.append(
        AIMessage(content=answer)
    )

    print("\n🤖 Assistant:")
    print(answer)

    print("\n📚 Chat History Length:",
          len(chat_history))



d:\github\Learning\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Administrator\AppData\Local\Temp\ipykernel_28120\649519890.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


🔄 Initializing Ollama (Llama 3.1)...
🔄 Loading Embedding Model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8215.68it/s]


🔄 Connecting ChromaDB...


C:\Users\Administrator\AppData\Local\Temp\ipykernel_28120\649519890.py:54: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


In [2]:
# ==========================================
# 11. Test
# ==========================================

run_rag_pipeline(
    "Who is Abu Sufiun?"
)




📝 User: Who is Abu Sufiun?

🤖 Assistant:
Abu Sufiun is a Full Stack Developer with expertise in Laravel & React.

📚 Chat History Length: 2


In [3]:
run_rag_pipeline(
    "How many skills does he have?"
)




📝 User: How many skills does he have?

🤖 Assistant:
He has 11 skills listed: 

1. RESTful API
2. Laravel
3. React
4. Bootstrap
5. Tailwind CSS
6. MS-SQL
7. Oracle
8. My-SQL
9. OOP (Object-Oriented Programming)
10. PHP
11. JavaScript, C, AI, and ML

📚 Chat History Length: 4


In [4]:
run_rag_pipeline(
    "What is his latest project?"
)


📝 User: What is his latest project?

🤖 Assistant:
I don't know based on the provided context.

📚 Chat History Length: 6


In [5]:
run_rag_pipeline(
    "how many project has in this cv?"
)


📝 User: how many project has in this cv?

🤖 Assistant:
There are 2 projects mentioned in the CV:

1. Project Management System
2. (no specific name mentioned for the second one, it is part of his experience as a Lecturer and Mid-Level Laravel Developer)

📚 Chat History Length: 8
